# Gemma 4 CEFR Classification of Learner Writing
**Model:** Gemma 4 31B Instruct Q4_K_S (unsloth/gemma-4-31B-it-GGUF)  
**Served via:** LM Studio (localhost:1234)  
**Corpus:** Write & Improve Corpus  
**Task:** CEFR level classification with feedback  
**Conditions:** Zero-shot, 8-shot, 16-shot, 24-shot × Non-Bayesian + Bayesian  
**Levels:** A2, A2+, B1, B1+, B2, B2+, C1, C1+

In [1]:
# Imports
import subprocess
import json
import pandas as pd
import time
import random

In [2]:
# Configuration
LM_STUDIO_URL = "http://localhost:1234/v1/chat/completions"
MODEL_NAME = "gemma-4-31b-it"  # update if LM Studio shows a different model string

LEVELS = ["A2", "A2+", "B1", "B1+", "B2", "B2+", "C1", "C1+"]
LEVELS_STR = ", ".join(LEVELS)

MAX_TOKENS = 600  # adjust based on observed response length in testing

In [3]:
# System prompt with CEFR Companion Volume (2020) Overall Written Production descriptors
# Descriptors used: Overall Written Production (broad levels only, as plus levels
# are not always separately defined in the Companion Volume - only for some descriptor sets)

SYSTEM_PROMPT = """You are an expert in assessing written English produced by learners of English as a foreign language.
You will assign a CEFR level to learner writing samples using the following official CEFR descriptors for Overall Written Production.

CEFR Overall Written Production Descriptors (Council of Europe, 2020):

C2: Can produce clear, smoothly flowing, complex texts in an appropriate and effective style 
and a logical structure which helps the reader identify significant points.

C1: Can produce clear, well-structured texts of complex subjects, underlining the relevant 
salient issues, expanding and supporting points of view at some length with subsidiary points, 
reasons and relevant examples, and rounding off with an appropriate conclusion.
Can employ the structure and conventions of a variety of genres, varying the tone, style and 
register according to addressee, text type and theme.

B2: Can produce clear, detailed texts on a variety of subjects related to their field of 
interest, synthesising and evaluating information and arguments from a number of sources.

B1: Can produce straightforward connected texts on a range of familiar subjects within their 
field of interest, by linking a series of shorter discrete elements into a linear sequence.

A2: Can produce a series of simple phrases and sentences linked with simple connectors 
like "and", "but" and "because".

A1: Can give information about matters of personal relevance (e.g. likes and dislikes, family, 
pets) using simple words and basic expressions.
Can produce simple isolated phrases and sentences.

Plus levels represent a very strong competence at a level that does not yet reach the minimum 
standard for the next criterion level. Generally, features of the level above are starting to appear.

The levels you should assign are: A2, A2+, B1, B1+, B2, B2+, C1, C1+.

Always respond in valid JSON format with no markdown formatting or code blocks."""

In [4]:
# Load data
df_train = pd.read_json("data/train.jsonl", lines=True)
df_val = pd.read_json("data/val.jsonl", lines=True)
df_test = pd.read_json("data/test.jsonl", lines=True)

# Filter to target levels only (actually I already filtered in the sampling, but just to be on the safe side)
df_train = df_train[df_train["label"].isin(LEVELS)].reset_index(drop=True)
df_val = df_val[df_val["label"].isin(LEVELS)].reset_index(drop=True)
df_test = df_test[df_test["label"].isin(LEVELS)].reset_index(drop=True)

print(f"Train: {len(df_train)} texts")
print(f"Validation:  {len(df_val)} texts")
print(f"Test:  {len(df_test)} texts")
print("\nTrain level distribution:")
print(df_train["label"].value_counts().sort_index())
print("\nValidation level distribution:")
print(df_val["label"].value_counts().sort_index())
print("\nTest level distribution:")
print(df_test["label"].value_counts().sort_index())

Train: 3797 texts
Validation:  599 texts
Test:  604 texts

Train level distribution:
label
A2     142
A2+    573
B1     702
B1+    882
B2     699
B2+    430
C1     296
C1+     73
Name: count, dtype: int64

Validation level distribution:
label
A2      30
A2+    136
B1     128
B1+     95
B2     110
B2+     49
C1      37
C1+     14
Name: count, dtype: int64

Test level distribution:
label
A2      30
A2+    137
B1     129
B1+     96
B2     110
B2+     50
C1      37
C1+     15
Name: count, dtype: int64


In [5]:
# Pre-sample for each shot condition - 8 and 16 are true subsets of 24
def sample_few_shot_examples(df, n_per_class, levels=LEVELS, random_state=42, topic_col="topic"):
    """
    Sample balanced, topic-diverse examples from training set.
    Returns a flat list of dicts with 'text', 'label', 'topic'.
    """
    examples = []
    rng = random.Random(random_state)
    
    for level in levels:
        subset = df[df["label"] == level]
        n = min(n_per_class, len(subset))
        
        if topic_col in subset.columns and subset[topic_col].nunique() >= n:
            available_topics = subset[topic_col].unique().tolist()
            rng.shuffle(available_topics)
            chosen_topics = available_topics[:n]
            
            sampled_rows = []
            for topic in chosen_topics:
                topic_subset = subset[subset[topic_col] == topic]
                sampled_rows.append(topic_subset.sample(n=1, random_state=random_state))
            sampled = pd.concat(sampled_rows)
        else:
            sampled = subset.sample(n=n, random_state=random_state)
        
        for _, row in sampled.iterrows():
            examples.append({
                "text": row["text"],
                "label": level,
                "topic": row.get(topic_col, None)
            })
    
    return examples  # NOT shuffled yet — keep level-grouped for clean slicing

# Step 1: sample 3 per level (24 total), level-grouped (3 consecutive per level)
examples_24_grouped = sample_few_shot_examples(df_train, n_per_class=3)

# Step 2: build nested subsets by taking the first k examples PER LEVEL
def take_first_n_per_level(grouped_examples, n_per_class, levels=LEVELS):
    result = []
    for level in levels:
        level_examples = [ex for ex in grouped_examples if ex["label"] == level]
        result.extend(level_examples[:n_per_class])
    return result

examples_24_subset = take_first_n_per_level(examples_24_grouped, 3)  # = all 24
examples_16_subset = take_first_n_per_level(examples_24_grouped, 2)  # first 2 per level
examples_8_subset  = take_first_n_per_level(examples_24_grouped, 1)  # first 1 per level

# Step 3: shuffle each condition independently for prompt order
rng = random.Random(42)
examples_24 = examples_24_subset.copy(); rng.shuffle(examples_24)
examples_16 = examples_16_subset.copy(); rng.shuffle(examples_16)
examples_8  = examples_8_subset.copy();  rng.shuffle(examples_8)

print(f"Examples pool sizes: 8={len(examples_8)}, 16={len(examples_16)}, 24={len(examples_24)}")

# Verify topic diversity for each condition
for n_shots, examples_set in [(8, examples_8), (16, examples_16), (24, examples_24)]:
    print(f"\n{n_shots} Examples:")
    for level in LEVELS:
        topics_used = [ex["topic"] for ex in examples_set if ex["label"] == level]
        print(f"{level}: {len(topics_used)} examples, {len(set(topics_used))} unique topics")

Examples pool sizes: 8=8, 16=16, 24=24

8 Examples:
A2: 1 examples, 1 unique topics
A2+: 1 examples, 1 unique topics
B1: 1 examples, 1 unique topics
B1+: 1 examples, 1 unique topics
B2: 1 examples, 1 unique topics
B2+: 1 examples, 1 unique topics
C1: 1 examples, 1 unique topics
C1+: 1 examples, 1 unique topics

16 Examples:
A2: 2 examples, 2 unique topics
A2+: 2 examples, 2 unique topics
B1: 2 examples, 2 unique topics
B1+: 2 examples, 2 unique topics
B2: 2 examples, 2 unique topics
B2+: 2 examples, 2 unique topics
C1: 2 examples, 2 unique topics
C1+: 2 examples, 2 unique topics

24 Examples:
A2: 3 examples, 3 unique topics
A2+: 3 examples, 3 unique topics
B1: 3 examples, 3 unique topics
B1+: 3 examples, 3 unique topics
B2: 3 examples, 3 unique topics
B2+: 3 examples, 3 unique topics
C1: 3 examples, 3 unique topics
C1+: 3 examples, 3 unique topics


In [6]:
# LM Studio API call function (via curl subprocess workaround, using temp file)
import subprocess
import tempfile
import os

def call_lm_studio(messages, max_retries=3, base_delay=5):
    payload = {
        "model": MODEL_NAME,
        "messages": messages,
        "temperature": 0.0,
        "max_tokens": MAX_TOKENS
    }
    payload_json = json.dumps(payload)

    for attempt in range(1, max_retries + 1):
        tmp_path = None
        try:
            # Write payload to a temp file to avoid Windows command-line length limits
            with tempfile.NamedTemporaryFile(mode="w", suffix=".json", delete=False, encoding="utf-8") as tmp:
                tmp.write(payload_json)
                tmp_path = tmp.name

            result = subprocess.run(
                ["curl.exe", "-s", "-X", "POST", LM_STUDIO_URL,
                 "-H", "Content-Type: application/json",
                 "-d", f"@{tmp_path}"],
                capture_output=True, text=True, timeout=120
            )
            if result.returncode != 0:
                raise RuntimeError(f"curl failed: {result.stderr}")

            response_json = json.loads(result.stdout)
            return response_json["choices"][0]["message"]["content"].strip()

        except Exception as e:
            print(f"  Error (attempt {attempt}/{max_retries}): {e}")
            if attempt == max_retries:
                raise
            time.sleep(base_delay * attempt)

        finally:
            if tmp_path and os.path.exists(tmp_path):
                os.remove(tmp_path)

In [7]:
# JSON parsing helper
def parse_response(raw_response, essay_id, condition):
    """
    Parse model response as JSON.
    Returns dict with parsed fields or error info.
    """
    text = raw_response.strip()
    # Strip markdown code blocks if present
    if text.startswith("```"):
        text = text.split("```")[1]
        if text.startswith("json"):
            text = text[4:]
        text = text.strip()
    try:
        parsed = json.loads(text)
        parsed["essay_id"] = essay_id
        parsed["condition"] = condition
        return parsed
    except json.JSONDecodeError as e:
        return {
            "essay_id": essay_id,
            "condition": condition,
            "error": f"JSON parse error: {e}",
            "raw_response": raw_response
        }

In [8]:
# Prompt for classification request (used in all conditions)
CLASSIFICATION_REQUEST = (
    f"Assign a CEFR level to the following learner writing. "
    f"The possible levels are: {LEVELS_STR}.\n\n"
    "Respond ONLY with a JSON object in this exact format:\n"
    '{"cefr_level": "...", "level_justification": "...", "strengths": "...", "areas_for_improvement": "..."}\n\n'
    "Field definitions:\n"
    "- cefr_level: the CEFR level you are assigning\n"
    "- level_justification: why this level was assigned, referring to the CEFR descriptors\n"
    "- strengths: what the learner does well in this text\n"
    "- areas_for_improvement: what the learner could improve to reach a higher level\n\n"
    "Text to classify:\n\n"
)

In [9]:
# Zero-shot condition
def classify_zero_shot(text):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": CLASSIFICATION_REQUEST + text}
    ]
    return call_lm_studio(messages)

In [10]:
# Few-shot condition (non-Bayesian)
# Chain prompting (showing the examples to the model as if it was a chat between the user and assistant)
# All examples provided upfront as user/assistant turns, then target text
def classify_few_shot(text, examples):
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]
    
    for ex in examples:
        messages.append({
            "role": "user",
            "content": CLASSIFICATION_REQUEST + ex["text"]
        })
        # Assistant response for few-shot examples: label only (no feedback)
        # since W&I provides CEFR labels but no rater feedback
        messages.append({
            "role": "assistant",
            "content": json.dumps({"cefr_level": ex["label"]})
        })
    
    # Final request: target text (full response expected)
    messages.append({
        "role": "user",
        "content": CLASSIFICATION_REQUEST + text
    })
    return call_lm_studio(messages)

In [11]:
# Bayesian condition
# Examples fed sequentially; correct label provided as feedback after each one
# Model asked to update its assessment criteria before proceeding
def classify_bayesian(text, examples):
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]
    
    for i, ex in enumerate(examples):
        # User presents example text
        messages.append({
            "role": "user",
            "content": CLASSIFICATION_REQUEST + ex["text"]
        })
        # Model's prediction (label only for examples)
        messages.append({
            "role": "assistant",
            "content": json.dumps({"cefr_level": ex["label"]})
        })
        # User provides correct label and asks model to update
        messages.append({
            "role": "user",
            "content": (
                f"The correct CEFR level for that text was {ex['label']}. "
                f"Please update your assessment criteria based on this {ex['label']} example "
                f"before proceeding to the next text."
            )
        })
        # Model acknowledges update
        messages.append({
            "role": "assistant",
            "content": (
                f"Understood. I have updated my assessment criteria based on this "
                f"{ex['label']} example and will apply this updated understanding "
                f"to subsequent texts."
            )
        })
    
    # Final request: target text (full response expected)
    messages.append({
        "role": "user",
        "content": (
            f"Based on your updated understanding from the {len(examples)} examples above, "
            + CLASSIFICATION_REQUEST + text
        )
    })
    return call_lm_studio(messages)

In [12]:
# Quick single-text test before running full experiment
# Use this to check the model is responding correctly and JSON is parseable

test_text = df_test.iloc[0]["text"]
test_id = df_test.iloc[0]["id"]
test_true = df_test.iloc[0]["label"]

print(f"Testing with: {test_id} (true label: {test_true})")
print("-" * 50)

raw = classify_zero_shot(test_text)
print("Raw response:")
print(raw)
print("\nParsed:")
print(json.dumps(parse_response(raw, test_id, "zero_shot"), indent=2))

Testing with: essay_28aba35720851372 (true label: B1+)
--------------------------------------------------
Raw response:
{
"cefr_level": "B1",
"level_justification": "The learner produces a straightforward connected text on a familiar subject (personal goals) by linking discrete elements into a linear sequence. While the structure is more of a list than a cohesive essay, it successfully communicates complex personal intentions and reasons, fitting the B1 descriptor for producing connected texts on familiar subjects.",
"strengths": "The learner can express abstract concepts such as 'innovative findings' and 'disseminate ideas'. They demonstrate a good range of vocabulary related to their professional field (statistics, algorithms, scientific articles) and can communicate their intentions clearly despite grammatical errors.",
"areas_for_improvement": "To reach B2, the learner needs to move beyond simple list-like structures toward more cohesive paragraphs with sophisticated linking words.

In [13]:
# Run all conditions
# Conditions: zero_shot, few_shot_8, few_shot_16, few_shot_24,
#             bayesian_8, bayesian_16, bayesian_24

few_shot_sets = {
    8:  examples_8,
    16: examples_16,
    24: examples_24
}

CONDITIONS = ["zero_shot"]
for n_shots in few_shot_sets:
    CONDITIONS += [f"few_shot_{n_shots}", f"bayesian_{n_shots}"]

    
all_results = []
total = len(df_test)

for idx, row in df_test.iterrows():
    text = row["text"]
    true_label = row["label"]
    essay_id = row["id"]
    
    print(f"\n[{idx+1}/{total}] {essay_id} | true: {true_label}")
    
    result = {
        "essay_id": essay_id,
        "true_label": true_label
    }
    
    # --- Zero-shot ---
    try:
        raw = classify_zero_shot(text)
        parsed = parse_response(raw, essay_id, "zero_shot")
        result["zero_shot_label"] = parsed.get("cefr_level", "ERROR")
        result["zero_shot_level_justification"] = parsed.get("level_justification", "")
        result["zero_shot_strengths"] = parsed.get("strengths", "")
        result["zero_shot_improvements"] = parsed.get("areas_for_improvement", "")
        result["zero_shot_raw"] = raw
        print(f"  zero_shot: {result['zero_shot_label']}")
    except Exception as e:
        result["zero_shot_label"] = "ERROR"
        result["zero_shot_raw"] = str(e)
        print(f"  zero_shot: ERROR - {e}")
    
    # --- Few-shot and Bayesian ---
    for n_shots, examples in few_shot_sets.items():
        
        # Non-Bayesian
        try:
            raw = classify_few_shot(text, examples)
            parsed = parse_response(raw, essay_id, f"few_shot_{n_shots}")
            result[f"few_shot_{n_shots}_label"] = parsed.get("cefr_level", "ERROR")
            result[f"few_shot_{n_shots}_level_justification"] = parsed.get("level_justification", "")
            result[f"few_shot_{n_shots}_strengths"] = parsed.get("strengths", "")
            result[f"few_shot_{n_shots}_improvements"] = parsed.get("areas_for_improvement", "")
            result[f"few_shot_{n_shots}_raw"] = raw
            print(f"  few_shot_{n_shots}: {result[f'few_shot_{n_shots}_label']}")
        except Exception as e:
            result[f"few_shot_{n_shots}_label"] = "ERROR"
            result[f"few_shot_{n_shots}_raw"] = str(e)
            print(f"  few_shot_{n_shots}: ERROR - {e}")
        
        # Bayesian
        try:
            raw = classify_bayesian(text, examples)
            parsed = parse_response(raw, essay_id, f"bayesian_{n_shots}")
            result[f"bayesian_{n_shots}_label"] = parsed.get("cefr_level", "ERROR")
            result[f"bayesian_{n_shots}_level_justification"] = parsed.get("level_justification", "")
            result[f"bayesian_{n_shots}_strengths"] = parsed.get("strengths", "")
            result[f"bayesian_{n_shots}_improvements"] = parsed.get("areas_for_improvement", "")
            result[f"bayesian_{n_shots}_raw"] = raw
            print(f"  bayesian_{n_shots}: {result[f'bayesian_{n_shots}_label']}")
        except Exception as e:
            result[f"bayesian_{n_shots}_label"] = "ERROR"
            result[f"bayesian_{n_shots}_raw"] = str(e)
            print(f"  bayesian_{n_shots}: ERROR - {e}")
    
    all_results.append(result)
    
    # Save results incrementally
    if (idx + 1) % 20 == 0:
        pd.DataFrame(all_results).to_csv("gemma4_cefr_results_partial_TEST.csv", index=False)
        print(f"  [Saved partial results: {idx + 1} texts processed]")
    
    # Small delay between texts to avoid overwhelming LM Studio
    time.sleep(1)

print(f"\n{'='*50}")
print(f"Complete. Processed {len(all_results)} texts.")


[1/604] essay_28aba35720851372 | true: B1+
  zero_shot: B1
  few_shot_8: B1
  bayesian_8: B1
  few_shot_16: B1
  bayesian_16: B1
  few_shot_24: B1
  bayesian_24: B1

[2/604] essay_2ae41ea2a3873e11 | true: B1
  zero_shot: B1
  few_shot_8: B1
  bayesian_8: B1
  few_shot_16: B1
  bayesian_16: B1
  few_shot_24: B1
  bayesian_24: B1

[3/604] essay_3924361a6a6c82d3 | true: A2+
  zero_shot: A2
  few_shot_8: B1
  bayesian_8: B1
  few_shot_16: B1
  bayesian_16: B1
  few_shot_24: B1
  bayesian_24: A2+

[4/604] essay_d6e366fddcd50ef1 | true: C1
  zero_shot: B2
  few_shot_8: B2
  bayesian_8: B2+
  few_shot_16: B2+
  bayesian_16: B2+
  few_shot_24: B2+
  bayesian_24: B2+

[5/604] essay_1cc8312707c22951 | true: A2+
  zero_shot: B1
  few_shot_8: B1
  bayesian_8: A2
  few_shot_16: A2+
  bayesian_16: A2
  few_shot_24: A2+
  bayesian_24: A2+

[6/604] essay_0e8a8b84336f54c9 | true: B2
  zero_shot: B1
  few_shot_8: B1+
  bayesian_8: B1+
  few_shot_16: B1+
  bayesian_16: B2
  few_shot_24: B2
  bayesian_24

In [14]:
# Save results
df_results = pd.DataFrame(all_results)

# Save full results including raw responses and feedback
df_results.to_csv("gemma4_cefr_results_TEST.csv", index=False)

# Save labels-only version for quick accuracy analysis
label_cols = ["essay_id", "true_label"] + [f"{c}_label" for c in CONDITIONS]

df_results[label_cols].to_csv("gemma4_cefr_results_labels_TEST.csv", index=False)

print("Saved:")
print("  gemma4_cefr_results_TEST.csv  (all fields including feedback)")
print("  gemma4_cefr_results_labels_TEST.csv (labels only for accuracy analysis)")
print(f"\nShape: {df_results.shape}")
df_results[label_cols].head()

Saved:
  gemma4_cefr_results_TEST.csv  (all fields including feedback)
  gemma4_cefr_results_labels_TEST.csv (labels only for accuracy analysis)

Shape: (604, 37)


,essay_id,true_label,zero_shot_label,few_shot_8_label,bayesian_8_label,few_shot_16_label,bayesian_16_label,few_shot_24_label,bayesian_24_label
0,essay_28aba35720851372,B1+,B1,B1,B1,B1,B1,B1,B1
1,essay_2ae41ea2a3873e11,B1,B1,B1,B1,B1,B1,B1,B1
2,essay_3924361a6a6c82d3,A2+,A2,B1,B1,B1,B1,B1,A2+
3,essay_d6e366fddcd50ef1,C1,B2,B2,B2+,B2+,B2+,B2+,B2+
4,essay_1cc8312707c22951,A2+,B1,B1,A2,A2+,A2,A2+,A2+


## 10 texts were actually classed as A1, none as A1+, but I also added that in
#### All 10 texts will be counted as A2, as it would not have been possible for encoder models to class as A1

In [22]:
def to_scale(s):
    return s.replace({"A1": "A2", "A1+": "A2"})

for c in CONDITIONS:
    df_results[f"{c}_label"] = to_scale(df_results[f"{c}_label"])

In [23]:
# ordinal metrics per condition (matches encoder reporting)
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score, cohen_kappa_score

level_order = {lvl: i for i, lvl in enumerate(LEVELS)}
label_conditions = [f"{c}_label" for c in CONDITIONS]
ALL_LABELS = list(range(len(LEVELS)))


def ordinal_metrics(yt, yp):
    yt = np.asarray(yt, dtype=int)
    yp = np.asarray(yp, dtype=int)
    diff = np.abs(yt - yp)
    return {
        "QWK": cohen_kappa_score(yt, yp, weights="quadratic", labels=ALL_LABELS),
        "MAE": diff.mean(),
        "Exact": (diff == 0).mean(),
        "Adj": (diff <= 1).mean(),
        "F1": f1_score(yt, yp, average="macro", labels=ALL_LABELS, zero_division=0),
    }


rows = []
for condition in label_conditions:
    valid = df_results[["true_label", condition]].copy()
    valid = valid[valid[condition].ne("ERROR")].dropna()
    yt = valid["true_label"].map(level_order)
    yp = valid[condition].map(level_order)
    keep = yt.notna() & yp.notna()
    dropped = int((~keep).sum())
    yt, yp = yt[keep].astype(int), yp[keep].astype(int)
    if len(yt) == 0:
        continue
    rows.append({"Condition": condition, "n": len(yt), **ordinal_metrics(yt, yp),
                 "unmapped": dropped})

df_metrics = pd.DataFrame(rows).set_index("Condition")
print(df_metrics.round(3).to_string())

                     n    QWK    MAE  Exact    Adj     F1  unmapped
Condition                                                          
zero_shot_label    604  0.550  1.228  0.232  0.661  0.113         0
few_shot_8_label   604  0.677  0.930  0.315  0.791  0.209         0
bayesian_8_label   604  0.639  0.965  0.318  0.776  0.208         0
few_shot_16_label  604  0.775  0.786  0.389  0.839  0.298         0
bayesian_16_label  604  0.775  0.780  0.409  0.834  0.314         0
few_shot_24_label  604  0.788  0.780  0.384  0.849  0.285         0
bayesian_24_label  604  0.778  0.816  0.366  0.834  0.279         0


# Confusion matrices

In [24]:
# Cell: confusion matrices from df_results (Plotly)
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.metrics import confusion_matrix

label_to_idx = {lvl: i for i, lvl in enumerate(LEVELS)}

def confusion_for_condition(df, condition):
    """Counts + row-normalized confusion matrix for one *_label column."""
    pred_col = f"{condition}_label"
    # isin(LEVELS) drops ERROR and any unparseable / out-of-set preds
    mask = df["true_label"].isin(LEVELS) & df[pred_col].isin(LEVELS)
    sub = df.loc[mask]
    y_true = sub["true_label"].map(label_to_idx).to_numpy()
    y_pred = sub[pred_col].map(label_to_idx).to_numpy()
    n = len(LEVELS)
    counts = confusion_matrix(y_true, y_pred, labels=list(range(n)))
    row_sums = counts.sum(axis=1, keepdims=True)
    with np.errstate(divide="ignore", invalid="ignore"):
        norm = np.divide(counts, row_sums, where=row_sums != 0)
    return counts, np.nan_to_num(norm)

def _annot(counts, norm):
    a = np.empty_like(counts, dtype=object)
    for i in range(counts.shape[0]):
        for j in range(counts.shape[1]):
            a[i, j] = f"{norm[i, j] * 100:.1f}%<br>({counts[i, j]})"
    return a

def plot_confusion_grid(conditions, n_rows, n_cols, title, subplot_titles=None):
    fig = make_subplots(
        rows=n_rows, cols=n_cols,
        subplot_titles=subplot_titles or conditions,
        horizontal_spacing=0.08, vertical_spacing=0.12,
    )
    fig.update_layout(coloraxis=dict(
        colorscale="Greys", cmin=0, cmax=1,
        colorbar=dict(title="Row-normalized"),
    ))
    for k, cond in enumerate(conditions):
        r, c = divmod(k, n_cols)
        r, c = r + 1, c + 1
        counts, norm = confusion_for_condition(df_results, cond)
        fig.add_trace(
            go.Heatmap(
                z=norm, x=LEVELS, y=LEVELS,
                text=_annot(counts, norm), texttemplate="%{text}",
                hoverongaps=False, coloraxis="coloraxis",
            ),
            row=r, col=c,
        )
        fig.update_xaxes(title_text="Predicted", row=r, col=c)
        fig.update_yaxes(title_text="True", autorange="reversed", row=r, col=c)
    fig.update_layout(
        title=dict(text=title),
        font=dict(family="Arial", size=16, color="black"),
        width=500 * n_cols, height=480 * n_rows,
        margin=dict(l=80, r=100, t=120, b=80),
    )
    return fig

# Zero shot

In [25]:
fig0 = plot_confusion_grid(
    ["zero_shot"], n_rows=1, n_cols=1, title="Zero-shot baseline",
)
fig0.show()

# Few-shot and Bayesian

In [27]:
fig = plot_confusion_grid(
    ["few_shot_8",  "bayesian_8",
     "few_shot_16", "bayesian_16",
     "few_shot_24", "bayesian_24"],
    n_rows=3, n_cols=2,
    title="Few-shot (left) vs Bayesian (right), by N",
)
fig.show()

# Is Gemma shooting for the middle?

In [28]:
# Cell: ridgeline of predicted-label histograms (Plotly)
import numpy as np
import plotly.graph_objects as go

# top-to-bottom display order; label -> df_results column
ridge_rows = [
    ("True labels",      "true_label"),
    ("Zero-shot",        "zero_shot_label"),
    ("8-shot",           "few_shot_8_label"),
    ("8-shot Bayesian",  "bayesian_8_label"),
    ("16-shot",          "few_shot_16_label"),
    ("16-shot Bayesian", "bayesian_16_label"),
    ("24-shot",          "few_shot_24_label"),
    ("24-shot Bayesian", "bayesian_24_label"),
]

idx = {lvl: i for i, lvl in enumerate(LEVELS)}
n_levels = len(LEVELS)

def level_proportions(series):
    s = series[series.isin(LEVELS)]                       # drops ERROR / out-of-set
    counts = s.map(idx).value_counts().reindex(range(n_levels), fill_value=0)
    total = counts.sum()
    return (counts / total).to_numpy() if total else counts.to_numpy().astype(float)

def row_color(col):
    if col == "true_label":         return "rgba(90,90,90,0.55)"
    if col == "zero_shot_label":    return "rgba(70,130,80,0.55)"
    if col.startswith("few_shot"):  return "rgba(70,110,170,0.55)"
    if col.startswith("bayesian"):  return "rgba(200,90,80,0.55)"
    return "rgba(120,120,120,0.55)"

spacing = 1.0     # vertical gap between baselines
scale   = 1.3     # full-proportion bar height in baseline units -> controls overlap

fig = go.Figure()
n_rows = len(ridge_rows)

for i, (name, col) in enumerate(ridge_rows):
    base = (n_rows - 1 - i) * spacing                     # top row sits highest
    p = level_proportions(df_results[col])

    xs, ys = [], []
    for j in range(n_levels):                             # stepped histogram top
        xs += [j - 0.5, j + 0.5]
        ys += [base + scale * p[j]] * 2
    poly_x = [-0.5] + xs + [n_levels - 0.5]               # close down to baseline
    poly_y = [base] + ys + [base]

    fig.add_trace(go.Scatter(
        x=poly_x, y=poly_y, fill="toself", mode="lines",
        line=dict(color="black", width=1), fillcolor=row_color(col),
        hoverinfo="skip", showlegend=False,
    ))

    mean_idx = float(np.dot(np.arange(n_levels), p))      # central-tendency marker
    fig.add_trace(go.Scatter(
        x=[mean_idx], y=[base], mode="markers",
        marker=dict(color="black", size=10, symbol="line-ns-open"),
        hoverinfo="skip", showlegend=False,
    ))

# dashed anchor at the TRUE-label mean, so drift toward centre is visible
true_mean = float(np.dot(np.arange(n_levels), level_proportions(df_results["true_label"])))
fig.add_vline(x=true_mean, line=dict(color="black", width=1, dash="dash"))

fig.update_layout(
    title="Predicted CEFR-level distributions by condition",
    xaxis=dict(tickmode="array", tickvals=list(range(n_levels)), ticktext=LEVELS,
               title="CEFR level", range=[-0.6, n_levels - 0.4]),
    yaxis=dict(tickmode="array",
               tickvals=[(n_rows - 1 - i) * spacing for i in range(n_rows)],
               ticktext=[name for name, _ in ridge_rows],
               showgrid=False, zeroline=False),
    height=105 * n_rows + 120, width=900,
    font=dict(family="Arial", size=14, color="black"),
    plot_bgcolor="white",
)
fig.show()

### Wilcoxon signed rank test & Sign test (exact binomial on the discordant pairs)
### Comparing few-shot versus Bayesian (at each few-shot level)
#### Not sure whcih test is more suitable, but the results are basically the same

In [29]:
from scipy.stats import wilcoxon, binomtest
from statsmodels.stats.multitest import multipletests

level_order = {lvl: i for i, lvl in enumerate(LEVELS)}
def dist_series(cond):
    return (df_results[f"{cond}_label"].map(level_order)
            - df_results["true_label"].map(level_order)).abs()

rows, sign_p, wilcox_p = [], [], []
for n in (8, 16, 24):
    diff = (dist_series(f"bayesian_{n}") - dist_series(f"few_shot_{n}")).dropna()
    bayes_better = int((diff < 0).sum())     # smaller distance = closer to truth
    few_better   = int((diff > 0).sum())
    tied         = int((diff == 0).sum())
    discordant   = bayes_better + few_better

    # Sign test: among discordant pairs, is the split different from 50/50?
    sp = binomtest(bayes_better, discordant, p=0.5, alternative="two-sided").pvalue

    # Wilcoxon signed-rank on the same paired differences (drops zeros)
    wp = wilcoxon(diff, zero_method="wilcox").pvalue

    rows.append((n, bayes_better, few_better, tied, discordant))
    sign_p.append(sp)
    wilcox_p.append(wp)

sign_holm   = multipletests(sign_p,   method="holm")[1]
wilcox_holm = multipletests(wilcox_p, method="holm")[1]

print(f"{'N':>3} | {'bayes':>5} {'few':>4} {'tied':>4} | "
      f"{'sign p':>8} {'sign Holm':>9} | {'wilcox p':>8} {'wilcox Holm':>11}")
print("-" * 70)
for (n, b, f, t, d), sp, sh, wp, wh in zip(rows, sign_p, sign_holm, wilcox_p, wilcox_holm):
    print(f"{n:>3} | {b:>5} {f:>4} {t:>4} | {sp:>8.4f} {sh:>9.4f} | {wp:>8.4f} {wh:>11.4f}")

  N | bayes  few tied |   sign p sign Holm | wilcox p wilcox Holm
----------------------------------------------------------------------
  8 |    22   43  539 |   0.0125    0.0375 |   0.0092      0.0276
 16 |    49   43  512 |   0.6024    0.6024 |   0.6879      0.6879
 24 |    33   54  517 |   0.0314    0.0628 |   0.0204      0.0408
